In [0]:
spark.sql("USE CATALOG 03_gold")
spark.sql("USE SCHEMA default")

In [0]:
spark.sql("SHOW TABLES").show()

In [0]:
spark.sql("""
SELECT SUM(total_sales) AS total_sales
FROM fact_sales
""").show()

In [0]:
df = spark.read.table("01_bronze.default.raw_sales_dtl")
print(df.columns)

In [0]:
spark.sql("""
SELECT product_id, SUM(total_profit)
FROM 03_gold.default.fact_sales
GROUP BY product_id
""").show()

In [0]:
spark.sql("""
SELECT p.category, SUM(f.total_sales) AS revenue
FROM fact_sales f
JOIN dim_product p
ON f.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC
""").show()

In [0]:
spark.sql("""
SELECT AVG(quantity) AS avg_basket_size
FROM fact_sales
""").show()

In [0]:
spark.sql("""
SELECT 
(COUNT(r.return_id) * 1.0 / COUNT(f.transaction_id)) * 100 AS return_rate
FROM 03_gold.default.fact_sales f
LEFT JOIN 02_silver.default.returns r
ON f.transaction_id = r.transaction_id
""").show()

In [0]:
spark.sql("""
SELECT COUNT(DISTINCT product_id) AS out_of_stock
FROM 02_silver.default.inventory
WHERE stock_qty = 0
""").show()

In [0]:
spark.sql("""
SELECT store_id,
SUM(total_sales) AS total_sales,
RANK() OVER (ORDER BY SUM(total_sales) DESC) AS rank
FROM fact_sales
GROUP BY store_id
""").show()

In [0]:
spark.sql("""
SELECT SUM(discount * quantity) AS discount_loss
FROM fact_sales
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS repeat_customers
FROM (
    SELECT customer_id
    FROM fact_sales
    GROUP BY customer_id
    HAVING COUNT(*) > 1
)
""").show()

In [0]:
spark.sql("""
SELECT p.product_id
FROM dim_product p
LEFT JOIN fact_sales f
ON p.product_id = f.product_id
AND f.transaction_date >= DATE_SUB(CURRENT_DATE(), 30)
WHERE f.product_id IS NULL
""").show()